# FPL 2023-24 Player Performance Analysis
This notebook performs end-to-end analysis of real-world Fantasy Premier League data, covering data cleaning, transformation, and exploratory visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a visually clear theme utilizing high luminance contrast for readability
sns.set_theme(style="whitegrid")

---
## Phase 1: Data Cleaning & Preparation

In this phase we:
1. **Fetch real-world data** directly from GitHub (2023-24 Premier League Stats).
2. **Clean anomalies & handle missing data** — players with 0 minutes are removed to prevent divide-by-zero errors, and remaining NaNs in numeric columns are imputed with the median (robust to extreme outliers).
3. **Derive new attributes** to reduce cognitive load — numeric positions are mapped to readable strings (supporting both numeric codes 1–4 and text abbreviations like GK/DEF/MID/FWD), and composite metrics (Goal Involvement, Points Per 90, Cost in Millions) are computed.
4. **Normalize** Total Points to a 0–1 scale for comparative analytics.

In [ ]:
print("--- Phase 1: Real-World Data Extraction & Cleaning ---")

# 1. Fetching REAL-WORLD DATA directly from GitHub (2023-24 Premier League Stats)
url = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2023-24/cleaned_players.csv"
df = pd.read_csv(url)

# 2. Data Cleaning: Filtering Anomalies & Handling Missing Data
# Filter out players with 0 minutes to prevent "Divide by Zero" anomalies later
df = df[df['minutes'] > 0].copy()
# Impute any remaining NaNs in numeric columns with the median (robust to extreme outliers)
numeric_cols = ['total_points', 'goals_scored', 'assists', 'now_cost']
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# 3. Data Transformation: Deriving New Attributes to Reduce Cognitive Load
# Support both numeric codes (1-4) and standard text abbreviations ('GK', 'DEF', 'MID', 'FWD')
pos_map = {
    1: 'Goalkeeper', 2: 'Defender', 3: 'Midfielder', 4: 'Forward',
    '1': 'Goalkeeper', '2': 'Defender', '3': 'Midfielder', '4': 'Forward',
    'GK': 'Goalkeeper', 'DEF': 'Defender', 'MID': 'Midfielder', 'FWD': 'Forward',
    'GKP': 'Goalkeeper'
}
df['Position'] = df['element_type'].map(pos_map)

# If some positions are still unmapped, keep the original string or fallback
if df['Position'].isna().all():
    df['Position'] = df['element_type']

# Deriving new attributes to abstract complex raw data into simple performance metrics
df['Goal_Involvement'] = df['goals_scored'] + df['assists']
df['Points_Per_90'] = (df['total_points'] / df['minutes']) * 90
df['Cost_Millions'] = df['now_cost'] / 10  # FPL costs are stored multiplied by 10

# 4. Data Normalization
# Scaling Total Points to a 0-1 scale for comparative analytics
df['Points_Normalized'] = (df['total_points'] - df['total_points'].min()) / (df['total_points'].max() - df['total_points'].min())

print("Data Cleaning Complete. Anomalies filtered, missing values handled, attributes derived and normalized.")
print(f"Unique values in element_type: {df['element_type'].unique()}")
print(f"Unique positions mapped: {df['Position'].unique()}")
print(f"Total non-null rows remaining: {len(df.dropna(subset=['Position']))}")

---
## Phase 2: Exploratory Data Analysis & Visualization

Four visualizations are produced, each chosen for a specific analytical purpose:

1. **Correlation Heatmap** — High-level overview of relationships between key metrics before drilling down.
2. **Box Plot** — Distribution and outlier detection per position; whisker fences expose superstar performers.
3. **Bar Plot** — Precise quantitative comparison of average points across positions using aligned length encoding.
4. **FacetGrid (Small Multiples)** — Side-by-side cost vs. performance scatter plots per position; redundant encoding (hue + shape) accommodates color deficiency.

In [ ]:
print("--- Phase 2: Exploratory Data Analysis & Visualization ---")

# Filter out any lingering NaN positions
plot_df = df.dropna(subset=['Position'])

# 1. Correlation Heatmap (The Overview)
# Provides a high-level summary of relationships before detailed analysis.
plt.figure(figsize=(8, 6))
corr_matrix = plot_df[['Cost_Millions', 'total_points', 'minutes', 'Goal_Involvement', 'Points_Per_90']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, center=0)
plt.title('EDA 1: Correlation Matrix of Player Metrics', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

# 2. Box Plot (Anomaly & Distribution Detection)
# Static item aggregation tuned perfectly for finding distributions and isolating outliers beyond the fences.
plt.figure(figsize=(10, 5))
sns.boxplot(x='total_points', y='Position', data=plot_df, hue='Position', palette='Set2', legend=False)
plt.title('EDA 2: Distribution and Outliers of Total Points by Position', fontsize=14, weight='bold')
plt.xlabel('Total Season Points')
plt.ylabel('Tactical Position')
# Annotating the visual to explicitly point out the messy, real-world outliers
plt.text(plot_df['total_points'].max() - 20, 3.2, 'Statistical Outliers\n(Superstar Performers)',
         color='red', weight='bold', ha='center')
plt.tight_layout()
plt.show()

# 3. Bar Plot (High-Accuracy 2D Spatial Position)
# Relies on aligned length/position for precise quantitative comparison across categories.
plt.figure(figsize=(10, 5))
sns.barplot(x='total_points', y='Position', data=plot_df, estimator=np.mean, errorbar=None, palette='viridis', hue='Position', legend=False)
plt.title('Viz 1: Average Total Points by Tactical Position', fontsize=14, weight='bold')
plt.xlabel('Average Total Points')
plt.ylabel('Player Position')
plt.tight_layout()
plt.show()

# 4. FacetGrid (Small Multiples / Eyes Beat Memory)
# Juxtaposing views side-by-side to compare categorical roles without relying on cognitive memory.
# Uses redundant encoding (hue + shape) to accommodate color deficiency.
g = sns.FacetGrid(plot_df, col="Position", col_wrap=2, height=4, aspect=1.2)
g.map_dataframe(sns.scatterplot, x="Cost_Millions", y="total_points", hue="Position", style="Position", s=60, palette='colorblind')
g.set_axis_labels("Player Cost (Millions £)", "Total Season Points")
g.set_titles(col_template="{col_name}")
g.fig.suptitle('Viz 2: Cost vs. Performance Faceted by Position', fontsize=16, weight='bold', y=1.05)
plt.show()

---
## Phase 3: Data Analytics — K-Means Clustering

Now that the data is cleaned and we have visually explored the distributions, we move into **Phase 3: Data Analytics**.

Rather than relying on official positions (Forward, Defender, etc.), we apply **K-Means Clustering** to group players purely based on their underlying statistical performance — Cost, Points Per 90, and Minutes Played. This directly applies course theory: clustering serves as a powerful **aggregation approach** that creates proxies for a whole set of items, allowing us to evaluate statistical measures across much more **homogeneous groups**.

The pipeline is:
1. **Feature selection & filtering** — retain only players with ≥ 60 minutes played, ensuring `Points_Per_90` is statistically valid and free from scale anomalies.
2. **Standardization** — scale to Mean=0, Variance=1 so `minutes` (thousands) does not overpower `Cost` (tens) during distance calculations.
3. **K-Means algorithm** — partition players into 4 clusters (`n_clusters=4`).
4. **Cluster summary & visualization** — mean stats per cluster printed, then plotted as a scatter using color (hue) to encode cluster membership.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("--- Phase 3: Data Analytics (K-Means Clustering) - REVISED ---")

# 1. Feature Selection & Filtering Anomalies
# We filter out players with < 90 minutes so 'Points_Per_90' is statistically valid
df_cluster = df[df['minutes'] >= 60].copy()

features = ['Cost_Millions', 'Points_Per_90', 'minutes']
X = df_cluster[features]

# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Applying K-Means Clustering
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_cluster['Performance_Cluster'] = kmeans.fit_predict(X_scaled)

# 3. Labeling the Clusters based on Cost and Minutes
# Instead of hardcoding names, we will let the summary speak for itself first
cluster_means = df_cluster.groupby('Performance_Cluster')[['Cost_Millions', 'Points_Per_90', 'minutes']].mean().round(2)
print("\nRevised Cluster Analytics Summary (Min. 90 minutes played):")
print(cluster_means)

# 4. Data Visualization of the Analytics
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Cost_Millions', y='Points_Per_90', hue='Performance_Cluster',
                palette='Set1', data=df_cluster, s=80)

plt.title('K-Means Clustering: Player Segmentation (Filtered > 90 mins)', fontsize=14, weight='bold')
plt.xlabel('Cost (Millions £)')
plt.ylabel('Fantasy Points Per 90 Minutes')
plt.legend(title='Cluster ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()